In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Banking Data Analysis Silver Layer").getOrCreate()

In [0]:
accounts = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Account_Data.csv", header = True, inferSchema = True)
branches = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Branch_Data.csv", header = True, inferSchema = True)
customers = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Customer_Data.csv", header = True, inferSchema = True)
loans = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Loan_Data.csv", header = True, inferSchema = True)
transactions = spark.read.csv("/Volumes/workspace/default/banking_data_analysis/Bronze_Transacation_Data.csv", header = True, inferSchema = True)

In [0]:
display(accounts.limit(10))
display(branches.limit(10))
display(customers.limit(10))
display(loans.limit(10))
display(transactions.limit(10))

In [0]:
accounts.printSchema()
branches.printSchema()
customers.printSchema()
loans.printSchema()
transactions.printSchema()

## cleaning customers

In [0]:
median_phone = customers \
    .withColumn("Phone_Number",
        F.expr("try_cast(regexp_replace(Phone_Number, '[^0-9]', '') as BIGINT)")
    ) \
    .filter(F.col("Phone_Number").isNotNull()) \
    .approxQuantile("Phone_Number", [0.5], 0.01)[0]

In [0]:
clean_phone = F.expr(
    "try_cast(regexp_replace(Phone_Number, '[^0-9]', '') as BIGINT)"
)

silver_customers = (
    customers

    # 1. Drop rows where primary key is missing
    .filter(F.col("CUSTOMER_ID").isNotNull())

    # 2. Standardise CUSTOMER_ID
    .withColumn("CUSTOMER_ID", F.upper(F.trim(F.col("CUSTOMER_ID"))))

    # 3. Names
    .withColumn("First_Name",
        F.when(F.col("First_Name").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("First_Name")))))
    .withColumn("Last_Name",
        F.when(F.col("Last_Name").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("Last_Name")))))

    # 4. City
    .withColumn("City",
        F.when(F.col("City").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("City")))))

    # 5. Clean + impute phone number 
    .withColumn("Phone_Number", clean_phone)
    .withColumn("Phone_Number",
        F.when(F.col("Phone_Number").isNull(), F.lit(median_phone))
         .otherwise(F.col("Phone_Number"))
    )

    # 6. Occupation
    .withColumn("Occupation",
        F.when(F.col("Occupation").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("Occupation")))))

    # 7. cast and fill nulls with default DOB 
    .withColumn("DOB",F.expr("try_to_date(DOB, 'yyyy-MM-dd')"))
    .withColumn("DOB",F.coalesce(F.col("DOB"), F.lit("1900-01-01").cast("date")))

    # 8. Dedup
    .dropDuplicates()
    .dropDuplicates(["CUSTOMER_ID"])

    # 9. Audit
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
display(silver_customers.limit(10))
silver_customers.printSchema()

In [0]:
silver_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_customers")

In [0]:
silver_customers.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/banking_data_analysis/delta_customers")

## cleaning branch table

In [0]:
silver_branches = (
    branches

    # 1. Drop rows where primary key is missing
    .filter(F.col("BRANCH_ID").isNotNull())

    # 2. Standardise BRANCH_ID → uppercase + trim
    .withColumn("BRANCH_ID", F.upper(F.trim(F.col("BRANCH_ID"))))

    # 3. Trim BRANCH_NAME; fill nulls
    .withColumn("BRANCH_NAME",
        F.when(F.col("BRANCH_NAME").isNull(), "Unknown Branch")
         .otherwise(F.trim(F.col("BRANCH_NAME"))))

    # 4. Proper-case BRANCH_STATE; fill nulls
    .withColumn("BRANCH_STATE",
        F.when(F.col("BRANCH_STATE").isNull(), "Unknown")
         .otherwise(F.initcap(F.trim(F.col("BRANCH_STATE")))))

    # 5. Drop fully duplicate rows, then dedup on primary key
    .dropDuplicates()
    .dropDuplicates(["BRANCH_ID"])

    # 6. Audit columns
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
print("Branches →", silver_branches.count())
display(silver_branches.limit(10))

In [0]:
silver_branches.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_branches")
    
silver_branches.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/banking_data_analysis/delta_branches")

## cleaning accounts table

In [0]:
clean_balance = F.expr("""
    try_cast(
        regexp_replace(OPENING_BALANCE, '[^0-9.]', '') 
    as DOUBLE)
""")

median_balance = accounts \
    .withColumn("OPENING_BALANCE", clean_balance) \
    .filter(F.col("OPENING_BALANCE").isNotNull()) \
    .approxQuantile("OPENING_BALANCE", [0.5], 0.01)[0]

print(f"Median opening balance: {median_balance:,.0f}")

silver_accounts = (
    accounts

    # 1. Drop rows where primary key is missing
    .filter(F.col("ACCOUNT_ID").isNotNull())

    # 2. Standardise IDs → uppercase + trim
    .withColumn("ACCOUNT_ID",  F.upper(F.trim(F.col("ACCOUNT_ID"))))
    .withColumn("CUSTOMER_ID", F.upper(F.trim(F.col("CUSTOMER_ID"))))
    .withColumn("BRANCH_ID",   F.upper(F.trim(F.col("BRANCH_ID"))))

    # 3. Normalise ACCOUNT_TYPE: lowercase → title-case
    #    handles: "SAVINGS","savings","Savings" → "Savings"
    #             "FIXED DEPOSIT","fixed deposit" → "Fixed Deposit"
    .withColumn("ACCOUNT_TYPE",
        F.when(F.col("ACCOUNT_TYPE").isNull(), "Unknown")
         .otherwise(F.initcap(F.lower(F.trim(F.col("ACCOUNT_TYPE"))))))

    # 4. Normalise ACCOUNT_STATUS the same way
    #    handles: "ACTIVE","active","Active" → "Active"
    .withColumn("ACCOUNT_STATUS",
        F.when(F.col("ACCOUNT_STATUS").isNull(), "Unknown")
         .otherwise(F.initcap(F.lower(F.trim(F.col("ACCOUNT_STATUS"))))))

    # 5. Parse ACCOUNT_OPEN_DATE as DateType
    .withColumn("ACCOUNT_OPEN_DATE",F.when(F.col("ACCOUNT_OPEN_DATE").isin("null", "NULL", ""), None)
    .otherwise(F.expr("try_to_date(ACCOUNT_OPEN_DATE, 'yyyy-MM-dd')")))
    
    .withColumn("ACCOUNT_OPEN_DATE",
    F.coalesce(F.col("ACCOUNT_OPEN_DATE"), F.lit("1900-01-01").cast("date")))

    # 6. Impute OPENING_BALANCE with median; cast to FloatType
    .withColumn("OPENING_BALANCE", clean_balance)
    .withColumn("OPENING_BALANCE",F.when(F.col("OPENING_BALANCE").isNull(), F.lit(median_balance))
    .otherwise(F.col("OPENING_BALANCE")))

    # 7. Drop orphaned accounts (FK nulls — no parent customer/branch)
    .filter(F.col("CUSTOMER_ID").isNotNull())
    .filter(F.col("BRANCH_ID").isNotNull())

    # 8. Drop fully duplicate rows, then dedup on primary key
    .dropDuplicates()
    .dropDuplicates(["ACCOUNT_ID"])

    # 9. Audit columns
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
print("Accounts →", silver_accounts.count())
display(silver_accounts.limit(10))

In [0]:
silver_accounts.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_accounts")
    
silver_accounts.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/banking_data_analysis/delta_accounts")

## clean loans table

In [0]:
median_loan = loans \
    .withColumn("LOAN_AMOUNT", F.expr("try_cast(LOAN_AMOUNT as DOUBLE)")) \
    .filter(F.col("LOAN_AMOUNT").isNotNull()) \
    .approxQuantile("LOAN_AMOUNT", [0.5], 0.01)[0]

print(f"Median loan amount: {median_loan:,.0f}")

silver_loans = (
    loans

    # 1. Drop rows where primary key is missing
    .filter(F.col("LOAN_ID").isNotNull())

    # 2. Standardise IDs
    .withColumn("LOAN_ID",     F.upper(F.trim(F.col("LOAN_ID"))))
    .withColumn("CUSTOMER_ID", F.upper(F.trim(F.col("CUSTOMER_ID"))))
    .withColumn("BRANCH_ID",   F.upper(F.trim(F.col("BRANCH_ID"))))

    # 3. Clean + cast LOAN_AMOUNT safely
    .withColumn("LOAN_AMOUNT", F.expr("try_cast(LOAN_AMOUNT as DOUBLE)"))
    .withColumn("LOAN_AMOUNT", F.coalesce(F.col("LOAN_AMOUNT"), F.lit(median_loan)))

    # 4. Drop orphaned loans
    .filter(F.col("CUSTOMER_ID").isNotNull())
    .filter(F.col("BRANCH_ID").isNotNull())

    # 5. Remove duplicates
    .dropDuplicates()
    .dropDuplicates(["LOAN_ID"])

    # 6. Audit column (optional)
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
print("Loans →", silver_loans.count())
display(silver_loans.limit(10))

In [0]:
silver_loans.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_loans")
    
silver_loans.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/banking_data_analysis/delta_loans")

## clean transactions table

In [0]:
# ── Clean + compute median transaction amount ──────────────────────
median_txn = transactions \
    .withColumn("TRANSCATION_AMOUNT", F.expr("try_cast(TRANSCATION_AMOUNT as DOUBLE)")) \
    .filter(F.col("TRANSCATION_AMOUNT").isNotNull()) \
    .approxQuantile("TRANSCATION_AMOUNT", [0.5], 0.01)[0]

print(f"Median transaction amount: {median_txn:,.0f}")

# ── Window for latest record ───────────────────────────────
window_spec = Window.partitionBy("TRANSACTION_ID") \
                    .orderBy(F.col("TRANSACTION_DATE").desc())

silver_transactions = (
    transactions

    # 1. Drop null PK
    .filter(F.col("TRANSCATION_ID").isNotNull())

    # 2. Fix column names
    .withColumnRenamed("TRANSCATION_ID",     "TRANSACTION_ID")
    .withColumnRenamed("TRANSCATION_DATE",   "TRANSACTION_DATE")
    .withColumnRenamed("TRANSCATION_MEDIA",  "TRANSACTION_MEDIA")
    .withColumnRenamed("TRANSCATION_TYPE",   "TRANSACTION_TYPE")
    .withColumnRenamed("TRANSCATION_AMOUNT", "TRANSACTION_AMOUNT")

    # 3. Standardize IDs
    .withColumn("TRANSACTION_ID", F.upper(F.trim(F.col("TRANSACTION_ID"))))
    .withColumn("ACCOUNT_ID",     F.upper(F.trim(F.col("ACCOUNT_ID"))))

    #  4. Clean + parse DATE safely
    .withColumn("TRANSACTION_DATE",
        F.when(F.col("TRANSACTION_DATE").isin("null", "NULL", ""), None)
         .otherwise(F.expr("try_to_date(TRANSACTION_DATE, 'yyyy-MM-dd')"))
    )
    .withColumn("TRANSACTION_DATE",
        F.coalesce(F.col("TRANSACTION_DATE"), F.lit("1900-01-01").cast("date"))
    )

    #  5. Clean STRING columns
    # TRANSACTION_MEDIA
    .withColumn("TRANSACTION_MEDIA",
    F.when(
        (F.col("TRANSACTION_MEDIA").isNull()) | 
        (F.col("TRANSACTION_MEDIA").isin("null", "NULL", "")),
        "Unknown"
    ).otherwise(F.trim(F.col("TRANSACTION_MEDIA"))))

    # TRANSACTION_TYPE
    .withColumn("TRANSACTION_TYPE",
    F.when(
        (F.col("TRANSACTION_TYPE").isNull()) | 
        (F.col("TRANSACTION_TYPE").isin("null", "NULL", "")),
        "Unknown"
    ).otherwise(F.initcap(F.trim(F.col("TRANSACTION_TYPE")))))

    #  6. Clean + cast amount safely
    .withColumn("TRANSACTION_AMOUNT", F.expr("try_cast(TRANSACTION_AMOUNT as DOUBLE)"))
    .withColumn("TRANSACTION_AMOUNT",
        F.coalesce(F.col("TRANSACTION_AMOUNT"), F.lit(median_txn))
    )

    # 7. Drop orphan FK
    .filter(F.col("ACCOUNT_ID").isNotNull())

    #  8. Keep latest record per TRANSACTION_ID
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")

    # 9. Audit column (optional)
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

In [0]:
print("Transactions →", silver_transactions.count())
display(silver_transactions.limit(10))

In [0]:
silver_transactions.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("delta_transactions")
    
silver_transactions.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/banking_data_analysis/delta_transactions")

## Referential Integrity checks(check for Orphaned records)

In [0]:
checks = [
    ("account  → customer",  silver_accounts.join(silver_customers, "CUSTOMER_ID", "left_anti")),
    ("account  → branch",    silver_accounts.join(silver_branches,  "BRANCH_ID",   "left_anti")),
    ("loan     → customer",  silver_loans.join(silver_customers,    "CUSTOMER_ID", "left_anti")),
    ("loan     → branch",    silver_loans.join(silver_branches,     "BRANCH_ID",   "left_anti")),
    ("txn      → account",   silver_transactions.join(silver_accounts, "ACCOUNT_ID", "left_anti")),
]

print("\n=== Referential Integrity Report ===")
for label, orphans in checks:
    n = orphans.count()
    print(f"  {label:<25}  {'✓ OK' if n == 0 else f'⚠  {n} orphans'}")

## Referential Integrity Enforcement

In [0]:
# 1. Remove orphans from ACCOUNTS
silver_accounts_clean = (
    silver_accounts
    .join(silver_customers.select("CUSTOMER_ID"), "CUSTOMER_ID", "inner")
    .join(silver_branches.select("BRANCH_ID"),   "BRANCH_ID",   "inner")
)

print("Accounts after orphan removal →", silver_accounts_clean.count())
display(silver_accounts_clean.limit(10))

# 2. Remove orphans from LOANS
silver_loans_clean = (
    silver_loans
    .join(silver_customers.select("CUSTOMER_ID"), "CUSTOMER_ID", "inner")
    .join(silver_branches.select("BRANCH_ID"),   "BRANCH_ID",   "inner")
)

print("Loans after orphan removal →", silver_loans_clean.count())
display(silver_loans_clean.limit(10))

# 3. Remove orphans from TRANSACTIONS
silver_transactions_clean = (
    silver_transactions
    .join(silver_accounts_clean.select("ACCOUNT_ID"), "ACCOUNT_ID", "inner")
)

print("Transactions after orphan removal →", silver_transactions_clean.count())
display(silver_transactions_clean.limit(10))

In [0]:
# Check for Orphans AFTER Removal

checks_after = [
    ("account  → customer",  silver_accounts_clean.join(silver_customers, "CUSTOMER_ID", "left_anti")),
    ("account  → branch",    silver_accounts_clean.join(silver_branches,  "BRANCH_ID",   "left_anti")),
    ("loan     → customer",  silver_loans_clean.join(silver_customers,    "CUSTOMER_ID", "left_anti")),
    ("loan     → branch",    silver_loans_clean.join(silver_branches,     "BRANCH_ID",   "left_anti")),
    ("txn      → account",   silver_transactions_clean.join(silver_accounts_clean, "ACCOUNT_ID", "left_anti")),
]

print("\n=== Referential Integrity AFTER Removal ===")
for label, orphans in checks_after:
    n = orphans.count()
    print(f"  {label:<25}  {'✓ OK (0 orphans)' if n == 0 else f'⚠ {n} orphans still exist'}")